# 02 - Landing Zone CSV para Bronze Delta Lake

Este notebook le os arquivos CSV do bucket **landing-zone** no MinIO e grava cada tabela em formato **Delta Lake** no bucket **bronze**.

Fluxo executado:

`MinIO / landing-zone/*.csv -> Spark -> MinIO / bronze/<tabela> em Delta Lake`

## 1. Imports e variaveis

In [ ]:
import os

import boto3
from botocore.client import Config
from delta.tables import DeltaTable
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv(override=True)

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
LANDING_BUCKET = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')
BRONZE_BUCKET = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

tables = ['clientes', 'produtos', 'pedidos', 'itens_pedido']

print(f'MinIO: {MINIO_ENDPOINT}')
print(f'Landing: {LANDING_BUCKET} | Bronze: {BRONZE_BUCKET}')
print(f'Tabelas: {tables}')

## 2. Criar SparkSession com Delta Lake e MinIO

In [ ]:
spark = (
    SparkSession.builder
    .appName('Landing Zone CSV to Bronze Delta Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('SparkSession criada com suporte a Delta Lake e MinIO.')

## 3. Criar e limpar bucket bronze

In [ ]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] ja existe.')
except Exception:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado.')

response = s3_client.list_objects_v2(Bucket=BRONZE_BUCKET)
objetos = [{'Key': obj['Key']} for obj in response.get('Contents', [])]

if objetos:
    s3_client.delete_objects(Bucket=BRONZE_BUCKET, Delete={'Objects': objetos})
    print(f'{len(objetos)} objeto(s) removido(s) do bucket [{BRONZE_BUCKET}].')
else:
    print(f'Bucket [{BRONZE_BUCKET}] ja estava vazio.')

## 4. Validar arquivos CSV no landing-zone

In [ ]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
landing_keys = sorted(obj['Key'] for obj in response.get('Contents', []))
expected_keys = [f'{table}.csv' for table in tables]

print(f'Arquivos encontrados no bucket [{LANDING_BUCKET}]:')
for key in landing_keys:
    print(f'  - {key}')

missing_keys = [key for key in expected_keys if key not in landing_keys]
if missing_keys:
    raise FileNotFoundError(f'Arquivos ausentes no landing-zone: {missing_keys}')

print('Todos os CSVs esperados foram encontrados no landing-zone.')

## 5. Ler CSVs e gravar tabelas Delta no bronze

In [ ]:
for table in tables:
    print(f'Convertendo tabela {table} para Delta Lake...')

    input_path = f's3a://{LANDING_BUCKET}/{table}.csv'
    output_path = f's3a://{BRONZE_BUCKET}/{table}'

    df = (
        spark.read
        .option('header', 'true')
        .option('inferSchema', 'true')
        .csv(input_path)
    )

    total = df.count()
    print(f'Registros lidos de {table}: {total}')
    df.printSchema()

    (
        df.write
        .format('delta')
        .mode('overwrite')
        .save(output_path)
    )

    print(f'Tabela {table} salva em Delta Lake em: {output_path}\n')

print('Conversao para Delta Lake concluida.')

## 6. Validar tabelas Delta no bronze

In [ ]:
for table in tables:
    path = f's3a://{BRONZE_BUCKET}/{table}'

    print(f'Validando tabela Delta: {table}')
    is_delta = DeltaTable.isDeltaTable(spark, path)
    print(f'Eh Delta Lake: {is_delta}')

    df_delta = (
        spark.read
        .format('delta')
        .load(path)
    )

    print(f'Total de registros em {table}: {df_delta.count()}')
    df_delta.show(5, truncate=False)

    if not is_delta:
        raise RuntimeError(f'A tabela {table} nao foi gravada como Delta Lake.')

## 7. Validar _delta_log no bucket bronze

In [ ]:
for table in tables:
    prefix = f'{table}/_delta_log/'
    response = s3_client.list_objects_v2(Bucket=BRONZE_BUCKET, Prefix=prefix)
    delta_log_files = response.get('Contents', [])

    print(f'{table}: {len(delta_log_files)} arquivo(s) em {BRONZE_BUCKET}/{prefix}')

    if not delta_log_files:
        raise RuntimeError(f'_delta_log nao encontrado para a tabela {table}.')

## 8. Historico Delta

In [ ]:
for table in tables:
    path = f's3a://{BRONZE_BUCKET}/{table}'

    print(f'Historico Delta da tabela: {table}')
    history_df = spark.sql(f'DESCRIBE HISTORY delta.`{path}`')
    history_df.show(truncate=False)

## 9. Encerrar Spark

In [ ]:
spark.stop()
print('SparkSession finalizada.')